In [1]:
import json
import os
import sys
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
from pypdf import PdfReader

sys.path.insert(0, "..")

load_dotenv(find_dotenv(usecwd=True))
llm = OpenAI(base_url="http://localhost:4000", api_key=os.environ["LITELLM_MASTER_KEY"])
DATA = Path("../data")

In [2]:
def pdf_text(path: Path, mode: str = "layout") -> str:
    return "\n".join(
        page.extract_text(extraction_mode=mode) or "" for page in PdfReader(path).pages
    )


text = pdf_text(DATA / "pos" / "PO-0002.pdf")
i = text.find("Unterleg")
print(text[i - 200 : i + 400])

reis          Gesamtpreis

1                         Rillenkugellager 6205-2RS C3                                   17    PC                    8,75                  148,75

2        400103           Unterlegscheibe DIN 125 A 8,4 verzinkt, 100                    47    PAK                   3,20                  150,40
                          Stk

3                         Hydrauliköl HLP 46                                             20    L                     3,45                   69,00

4        400125           LED-Hallenstrahler 150 W IP65                                  25    PC     


In [4]:
from typing import Literal

from pydantic import BaseModel


class Classification(BaseModel):
    doc_type: Literal["purchase_order", "invoice", "remittance", "other"]
    reason: str


CLASSIFY_PROMPT = f"""You classify documents received by Alpentech Industriebedarf GmbH, a supplier.
Types:
- purchase_order: a customer orders goods FROM Alpentech.
- invoice: someone bills Alpentech or requests payment.
- remittance: a customer informs Alpentech about a payment.
- other: anything else (order confirmations, delivery notes, newsletters).
Return only JSON matching: {json.dumps(Classification.model_json_schema())}
"""


def classify(text: str, doc_id: str) -> Classification:
    response = llm.chat.completions.create(
        model="o2c-extract",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": CLASSIFY_PROMPT},
            {"role": "user", "content": text[:4000]},
        ],
        extra_body={"metadata": {"trace_name": "classify", "tags": ["poc", doc_id]}},
    )
    return Classification.model_validate_json(response.choices[0].message.content)

In [5]:
from collections import Counter

labels = json.loads((DATA / "doc_types.json").read_text(encoding="utf-8"))
results, mistakes = Counter(), []
for name, expected in labels.items():
    folder = "pos" if expected == "purchase_order" else "other"
    got = classify(pdf_text(DATA / folder / name), name)
    results[(expected, got.doc_type)] += 1
    if got.doc_type != expected:
        mistakes.append((name, expected, got.doc_type, got.reason))

correct = sum(n for (e, g), n in results.items() if e == g)
print(f"Accuracy: {correct}/{len(labels)}")
for m in mistakes:
    print(m)

Accuracy: 80/80
